# HarborGuard LoRA 训练（C4 · Colab · Unsloth + TRL SFT）

任务：把 C3 的「上下文 JSON → 建议 JSON」窄任务蒸馏到本地小模型，导出 GGUF 交给 Ollama 离线接管。

**运行前**：上传 `ml/lora/datasets/train.jsonl`（由 `python -m ml.lora.distill` 生成，450 条合格样本）。
可选上传 `test.jsonl` 用于快速冒烟。

训练口径（任务书 C4）：
- 基座：Qwen3.5 小尺寸（先 0.8B）；Unsloth/Ollama 兼容问题自动落到 **Qwen3 1.7B / 0.6B**（候选链见配置格）；
- LoRA **rank 16 / alpha 32**、lr **2e-4**、**2–3 epoch**、max_len **4096**；
- **bf16 LoRA，不做 4-bit 训练**（Unsloth 社区不建议对 Qwen3.5 做 4-bit 量化训练）；
- 只对 assistant 段计损失（prompt/completion 格式，切分用模型自己的 chat template）；
- 导出 **GGUF q4_k_m**（失败可换 q8_0），文件命名 `harborguard.gguf`；
- 固定 seed=0，与全项目一致。

系统提示词**不**进 Modelfile：部署后仍由请求发送（与 C3 运行时、蒸馏数据三方一致）。

In [ ]:
%pip install -q -U unsloth
%pip install -q -U trl datasets accelerate peft
# 装完如提示重启 runtime，先重启再从本格往下运行

In [ ]:
from pathlib import Path

# 基座候选：按序尝试，第一个能加载的生效。模型名以 Unsloth 文档为准
# （https://docs.unsloth.ai/get-started/all-our-models）；Qwen3.5 命名或兼容有问题时
# 自动落回 Qwen3（任务书 C4 兜底）。最终以本格打印的 BASE_USED 为准，
# 记入 ml/lora/metrics.json 的 base.model。
BASE_CANDIDATES = [
    "unsloth/Qwen3.5-0.8B",
    "unsloth/Qwen3-1.7B",
    "unsloth/Qwen3-0.6B",
]
MAX_LEN       = 4096
LORA_R        = 16
LORA_ALPHA    = 32      # 任务书允许 16–32
LR            = 2e-4
EPOCHS        = 3       # 任务书 2–3；时间紧可改 2
BATCH_DEVICES = 2
GRAD_ACCUM    = 8       # 有效 batch = 16
SEED          = 0       # 全项目固定种子
USE_4BIT      = False   # 任务书：bf16 LoRA，不做 4-bit 训练
TRAIN_JSONL   = "train.jsonl"
TEST_JSONL    = "test.jsonl"
assert Path(TRAIN_JSONL).exists(), "先上传 train.jsonl（见下一格）"
print("配置就绪：", dict(r=LORA_R, alpha=LORA_ALPHA, lr=LR, epochs=EPOCHS, max_len=MAX_LEN))

In [ ]:
# 已手动上传可跳过本格
if not Path(TRAIN_JSONL).exists():
    try:
        from google.colab import files
        print("请选择本机的 ml/lora/datasets/train.jsonl（可选再选 test.jsonl）")
        files.upload()
    except ImportError:
        print("非 Colab：把 train.jsonl 复制到当前目录")
assert Path(TRAIN_JSONL).exists(), "缺少 train.jsonl"
import json
rows = [json.loads(l) for l in open(TRAIN_JSONL, encoding="utf-8") if l.strip()]
assert all(len(r.get("messages", [])) == 3 for r in rows), "每行应为 system/user/assistant 三段"
systems = {r["messages"][0]["content"] for r in rows}
print(f"训练样本 {len(rows)} 条；system 提示词唯一: {len(systems) == 1}")

In [ ]:
from unsloth import FastLanguageModel

model = tokenizer = None
BASE_USED = None
for candidate in BASE_CANDIDATES:
    try:
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=candidate,
            max_seq_length=MAX_LEN,
            dtype=None,            # Unsloth 自动选择；bf16 训练由 SFTConfig(bf16=True) 控制
            load_in_4bit=USE_4BIT,
        )
        BASE_USED = candidate
        print("基座 =", candidate)
        break
    except Exception as exc:
        print(f"[跳过] {candidate}: {type(exc).__name__}: {str(exc)[:160]}")
assert model is not None, "候选基座全部失败：到 Unsloth 文档核对 Qwen3.5 模型名后更新 BASE_CANDIDATES"

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
model.print_trainable_parameters()

In [ ]:
from datasets import Dataset

def to_prompt_completion(messages):
    """prompt = system+user（含生成提示符），completion = assistant 段（含结束符）。
    切分用模型自己的 chat template，保证与推理请求格式完全一致。"""
    full = tokenizer.apply_chat_template(messages, tokenize=False)
    prompt = tokenizer.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True)
    assert full.startswith(prompt), "chat template 切分失败：更换基座候选或检查模板"
    return {"prompt": prompt, "completion": full[len(prompt):]}

train_ds = Dataset.from_list([to_prompt_completion(r["messages"]) for r in rows])
demo = train_ds[0]
print("PROMPT 尾部:", demo["prompt"][-160:])
print("COMPLETION:", demo["completion"][:200])

In [ ]:
import inspect
from trl import SFTConfig, SFTTrainer

cfg = dict(
    output_dir="harborguard-lora-out",
    per_device_train_batch_size=BATCH_DEVICES,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="no",
    bf16=True,
    seed=SEED,
    packing=False,
    dataset_num_proc=2,
    report_to="none",
)
# 兼容 TRL 版本差异：max_length（新）/ max_seq_length（旧），processing_class（新）/ tokenizer（旧）
params = inspect.signature(SFTConfig.__init__).parameters
cfg["max_length" if "max_length" in params else "max_seq_length"] = MAX_LEN

def build_trainer():
    tparams = inspect.signature(SFTTrainer.__init__).parameters
    key = "processing_class" if "processing_class" in tparams else "tokenizer"
    return SFTTrainer(model=model, train_dataset=train_ds, args=SFTConfig(**cfg), **{key: tokenizer})

trainer = build_trainer()
trainer.train()

In [ ]:
import time
FastLanguageModel.for_inference(model)

test_rows = []
if Path(TEST_JSONL).exists():
    test_rows = [json.loads(l) for l in open(TEST_JSONL, encoding="utf-8") if l.strip()]
else:
    test_rows = rows[-3:]  # 未上传 test.jsonl 时用训练样本尾部冒烟
demo_messages = test_rows[0]["messages"]
demo_prompt = tokenizer.apply_chat_template(demo_messages[:-1], tokenize=False, add_generation_prompt=True)
inputs = tokenizer(demo_prompt, return_tensors="pt").to(model.device)

started = time.perf_counter()
output = model.generate(**inputs, max_new_tokens=400, do_sample=False)
latency = time.perf_counter() - started
answer = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(f"Colab 生成耗时 {latency:.1f}s（验收：本机 ≤ 10s，此处仅作量级参考）\n")
print(answer)
import json as _json
try:
    parsed = _json.loads(answer)
    print("\nJSON 解析: OK；字段:", sorted(parsed))
except Exception as exc:
    print("\nJSON 解析失败:", exc)

In [ ]:
import glob, shutil

model.save_pretrained_merged("harborguard-merged", tokenizer, save_method="merged_16bit")
model.save_pretrained_gguf("harborguard_gguf", tokenizer, quantization_method="q4_k_m")
# 兼容问题改用 q8_0：model.save_pretrained_gguf("harborguard_gguf", tokenizer, quantization_method="q8_0")

src = sorted(glob.glob("harborguard_gguf/**/*.gguf", recursive=True))[0]
shutil.copy(src, "harborguard.gguf")
print(f"导出 harborguard.gguf（{Path('harborguard.gguf').stat().st_size / 1e9:.2f} GB）")

In [ ]:
try:
    from google.colab import files
    files.download("harborguard.gguf")
except ImportError:
    print("非 Colab：harborguard.gguf 已在当前目录")

## 下载后的本机部署（MacBook / 演示机）

1. 把 `harborguard.gguf` 放到仓库 `ml/lora/` 目录（与 `Modelfile` 同目录，GGUF 已被 gitignore）：

   ```bash
   cd ml/lora
   ollama create harborguard-lora -f Modelfile
   ollama run harborguard-lora "[{\"kind\":\"security\"}]"   # 冒烟（真实输入见 test.jsonl）
   ```

2. 验收与指标（基座 vs LoRA，50 条测试上下文）：

   ```bash
   python -m ml.lora.eval                 # 写 ml/lora/metrics.json
   curl -s localhost:8000/api/metrics     # 后端读取 {yolo, lora}
   ```

3. 断网验收：关掉 Wi-Fi 后 `ollama run harborguard-lora` 仍能输出；Agent 路由自动切本地。

### 排错 / 降级
- Qwen3.5 加载失败 → 配置格候选链已自动落到 Qwen3 1.7B / 0.6B，无需改代码；
- GGUF 导出或 Ollama 加载异常 → 换 `q8_0` 量化重导；仍失败时按任务书兜底：
  「基座模型 + 提示词」做本地路径（`OLLAMA_MODEL=qwen3:1.7b` 已可跑，指标卡如实显示）；
- 训练显存不足 → `BATCH_DEVICES=1` 并把 `GRAD_ACCUM` 提到 16；时间不足 → `EPOCHS=2`。